# [SOLUCIONARIO] Laboratorio 05 — Redes Convolucionales y Recurrentes Aplicadas
### Tecnologías Emergentes (ISO46B) — Semana 5 — Material del docente

Código completo con celdas de verificación automática (✅).


## Parte 1 — Básico: CNN de clasificación de imágenes

### Paso 1.1 — Cargar y explorar Fashion-MNIST

In [ ]:
from tensorflow import keras
import matplotlib.pyplot as plt

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

NOMBRES_CLASES = [
    "Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo",
    "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín",
]

print("Forma de X_train:", X_train.shape)
print("Forma de X_test:", X_test.shape)

In [ ]:
# ✅ Verificación automática
assert X_train.shape == (60000, 28, 28)
assert X_test.shape == (10000, 28, 28)
assert len(NOMBRES_CLASES) == 10
print("✅ Verificación 1.1 OK: Fashion-MNIST cargado con las formas esperadas.")

### Paso 1.2 — Normalizar y preparar los datos

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print("Forma final de X_train:", X_train.shape)

In [ ]:
# ✅ Verificación automática
assert X_train.shape == (60000, 28, 28, 1)
assert X_train.max() <= 1.0 and X_train.min() >= 0.0
print("✅ Verificación 1.2 OK: datos normalizados y remodelados correctamente.")

### Paso 1.3 — Construir la arquitectura CNN

In [ ]:
from tensorflow.keras import layers

modelo_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

modelo_cnn.summary()

In [ ]:
# ✅ Verificación automática
assert modelo_cnn.layers[-1].units == 10
assert modelo_cnn.layers[-1].activation.__name__ == "softmax"
print("✅ Verificación 1.3 OK: arquitectura correcta (salida softmax de 10 clases).")

### Paso 1.4 — Compilar, entrenar y evaluar

In [ ]:
modelo_cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

historial = modelo_cnn.fit(
    X_train, y_train,
    epochs=5, batch_size=64, validation_split=0.2, verbose=0,
)

perdida_test, precision_test = modelo_cnn.evaluate(X_test, y_test, verbose=0)
print(f"Precisión en test: {precision_test:.4f}")

In [ ]:
# ✅ Verificación automática
assert precision_test > 0.75, f"Precisión inusualmente baja ({precision_test:.2f})"
print("✅ Verificación 1.4 OK: precisión en test dentro de un rango razonable (>0.75).")

## Parte 2 — Intermedio: LSTM sobre la serie climática de Huancayo

### Paso 2.1 — Función de ventaneo

In [ ]:
import numpy as np
import pandas as pd


def crear_ventanas(serie, ventana):
    """Convierte una serie 1D en pares (X, y) usando una ventana deslizante."""
    X, y = [], []
    for i in range(len(serie) - ventana):
        X.append(serie[i : i + ventana])
        y.append(serie[i + ventana])
    return np.array(X), np.array(y)

In [ ]:
# ✅ Verificación automática
X_prueba, y_prueba = crear_ventanas([10, 11, 12, 13, 14], ventana=2)
assert X_prueba.shape == (3, 2)
assert list(X_prueba[0]) == [10, 11] and y_prueba[0] == 12
print("✅ Verificación 2.1 OK: crear_ventanas funciona correctamente.")

### Paso 2.2 — Cargar la serie y aplicar el ventaneo

In [ ]:
df = pd.read_csv("pronostico_huancayo.csv")
df = df.sort_values("fecha").reset_index(drop=True)
serie = df["temp_max"].values

VENTANA = 5
X, y = crear_ventanas(serie, VENTANA)
X = X.reshape(-1, VENTANA, 1)

print("X:", X.shape, " y:", y.shape)

### Paso 2.3 — Partición secuencial y escalado

In [ ]:
from sklearn.preprocessing import StandardScaler

corte = int(len(X) * 0.8)
X_train_serie, X_test_serie = X[:corte], X[corte:]
y_train_serie, y_test_serie = y[:corte], y[corte:]

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train_serie.reshape(-1, VENTANA)).reshape(-1, VENTANA, 1)
X_test_esc = escalador.transform(X_test_serie.reshape(-1, VENTANA)).reshape(-1, VENTANA, 1)

print("Entrenamiento:", X_train_esc.shape, " Prueba:", X_test_esc.shape)

In [ ]:
# ✅ Verificación automática
assert len(X_train_serie) + len(X_test_serie) == len(X)
assert abs(len(X_test_serie) / len(X) - 0.2) < 0.1
print("✅ Verificación 2.3 OK: partición secuencial dentro del rango esperado.")

### Paso 2.4 — Baseline ingenuo

In [ ]:
from sklearn.metrics import mean_absolute_error

prediccion_baseline = X_test_serie[:, -1].flatten()
mae_baseline = mean_absolute_error(y_test_serie, prediccion_baseline)
print(f"MAE del baseline ingenuo: {mae_baseline:.3f} °C")

### Paso 2.5 — Construir, compilar y entrenar la LSTM

In [ ]:
modelo_lstm = keras.Sequential([
    layers.Input(shape=(VENTANA, 1)),
    layers.LSTM(16, activation="tanh"),
    layers.Dense(1),
])

modelo_lstm.compile(optimizer="adam", loss="mse", metrics=["mae"])

historial_lstm = modelo_lstm.fit(
    X_train_esc, y_train_serie,
    epochs=50, batch_size=8, validation_split=0.2, verbose=0,
)

perdida_lstm, mae_lstm = modelo_lstm.evaluate(X_test_esc, y_test_serie, verbose=0)
print(f"MAE de la LSTM en test: {mae_lstm:.3f} °C")
print(f"MAE del baseline en test: {mae_baseline:.3f} °C")

In [ ]:
# ✅ Verificación automática
assert mae_lstm > 0
print("✅ Verificación 2.5 OK: LSTM entrenada y evaluada; comparación con baseline disponible.")
print(f"   Diferencia LSTM - baseline: {mae_lstm - mae_baseline:+.3f} °C")

**Respuesta de referencia:** con un dataset climático de menos de 100 días, es esperable que la LSTM apenas iguale o mejore levemente al baseline ingenuo. La ventaja de una LSTM se vuelve más clara con series más largas y con más variables de entrada (no solo temp_max).

## Parte 3 — Avanzado: sobreajuste, EarlyStopping y ajuste de hiperparámetros

### Paso 3.1 — Provocar sobreajuste deliberado

In [ ]:
X_pequeno, y_pequeno = X_train[:2000], y_train[:2000]


def construir_modelo():
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo


modelo_sobreajustado = construir_modelo()
historial_sin = modelo_sobreajustado.fit(
    X_pequeno, y_pequeno, epochs=25, validation_split=0.3, verbose=0,
)

plt.figure(figsize=(6, 4))
plt.plot(historial_sin.history["loss"], label="entrenamiento")
plt.plot(historial_sin.history["val_loss"], label="validación")
plt.title("SIN EarlyStopping")
plt.legend()
plt.show()

In [ ]:
# ✅ Verificación automática
assert len(historial_sin.history["loss"]) == 25
print("✅ Verificación 3.1 OK: entrenamiento sin EarlyStopping completó las 25 épocas programadas.")

### Paso 3.2 — Aplicar EarlyStopping

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

modelo_con_es = construir_modelo()
parada_temprana = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

historial_con = modelo_con_es.fit(
    X_pequeno, y_pequeno, epochs=25, validation_split=0.3,
    callbacks=[parada_temprana], verbose=0,
)

print(f"Entrenamiento detenido en la época {len(historial_con.history['loss'])} de 25 programadas.")

In [ ]:
# ✅ Verificación automática
assert len(historial_con.history["loss"]) <= 25
print("✅ Verificación 3.2 OK: EarlyStopping funcionó (entrenamiento no forzado a 25 épocas completas, o se detuvo antes según los datos).")

### Paso 3.3 — Experimentar con un hiperparámetro (ejemplo: número de filtros)

In [ ]:
def construir_modelo_variante(filtros=16):
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(filtros, kernel_size=3, activation="relu"),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])
    modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo


modelo_variante = construir_modelo_variante(filtros=16)
historial_variante = modelo_variante.fit(
    X_pequeno, y_pequeno, epochs=25, validation_split=0.3,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    verbose=0,
)

_, precision_base = modelo_con_es.evaluate(X_test, y_test, verbose=0)
_, precision_variante = modelo_variante.evaluate(X_test, y_test, verbose=0)

print(f"Precisión base (32 filtros): {precision_base:.4f}")
print(f"Precisión variante (16 filtros): {precision_variante:.4f}")

In [ ]:
# ✅ Verificación automática
assert 0.0 <= precision_base <= 1.0 and 0.0 <= precision_variante <= 1.0
print("✅ Verificación 3.3 OK: ambas variantes evaluadas; comparación disponible.")
print(f"   Diferencia (32 filtros - 16 filtros): {precision_base - precision_variante:+.4f}")

## Resumen para el docente

Todas las celdas ✅ deben ejecutarse sin `AssertionError`. La tabla de errores comunes del solucionario en `.docx` (Sección 2) cubre las causas más frecuentes observadas en semanas anteriores del curso, ahora extendida con los casos propios de CNN y LSTM.